# Chapter 6. Distributed Model Predictive Control — 분산 MPC

Rawlings, Mayne, Diehl (2017), **2판 1쇄 제6장, 인쇄 pp.369–450**의 한글 해설·실습입니다.
주요 절 6.1–6.7과 모든 하위 절 **23개**를 원본 순서로 배치합니다. 본문은 pp.369–434, 연습문제는 pp.435–447, 장 참고문헌은 pp.448–450입니다.

**학습 목표**
- 중앙집중·분권·비협조·협조 제어의 정보와 목적함수 차이를 구분한다.
- 선형 MPC의 상태 제거·KKT·리카티 해를 비교한다.
- 반복 횟수를 제한한 제어에서 입력 열의 기억과 비용 감소를 확인한다.
- 독립 입력 제약과 공유 자원 제약에서 협조 반복의 보장 차이를 설명한다.
- 비선형 비용에서 단순 평균이 실패하는 이유와 감소 검사 방법을 이해한다.
- 상태·외란 추정, 목표 계산, 유한 반복의 제어를 연결한다.

**실행:** Colab에서 `런타임 → 모두 실행`. 공통 준비부터 위에서 아래로 실행하세요. 앞서 정의한 모델·결과를 후속 실습에서 사용합니다.
이전 장의 실행이나 저장소 복제는 필요하지 않습니다. CPU를 사용하며 최초 설치·글꼴 다운로드에는 인터넷이 필요합니다.

**기호:** $k$는 실제 시스템의 시간 단계, $p$는 같은 시간 안에서의 최적화 반복, $N$은 예측 구간입니다.
$U$는 전체 예측 입력 열, $U_i$는 참여자 $i$가 소유하는 입력 열입니다. $\mathcal U_i$는 허용 집합입니다.
피드백은 $u=Kx$로 쓰며 $K$에 음의 부호를 포함합니다. 선형 LQ 비용은 공통 계수 1/2를 포함합니다.
비선형 MPC 실습에서는 1/2를 생략하며 코드와 설명에 같은 비용을 사용합니다.

**범위:** 원본 예제 6.9–6.12의 모델·설정과 반복 고유값을 재현하고, 예제 6.22의 초기 한 단계 비용을 분석합니다. 그 밖의 예는 직접 설계한 보충 실습입니다. §6.5.1–6.5.2는 교재 그림 6.7–6.8의 목적함수를 사용하되 상수 2를 더하고, 범위·시작점·종료 조건을 명시합니다.
교재의 모든 수치 예제·증명·연습문제 해답을 재현한 자료는 아닙니다. 서로 다른 절의 보충 모델은 각각 표시합니다.
여러 참여자의 계산을 한 노트북에서 순서대로 모사합니다. 실제 통신 지연·패킷 손실·병렬 실행 성능은 측정하지 않습니다.

[PPT](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter06_distributed_mpc/slides/chapter06_slides.pptx) · [PDF](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter06_distributed_mpc/slides/chapter06_slides.pdf)

## 실습 환경 준비

공통 선형 대수, 목적함수, 입력 열의 참여자별 인덱스와 한글 글꼴을 준비합니다.

In [ ]:
import importlib.util, subprocess, sys

packages = {
    "numpy": "numpy>=2,<3",
    "scipy": "scipy>=1.14,<2",
    "matplotlib": "matplotlib>=3.9,<4",
    "cvxpy": "cvxpy>=1.6,<2",
    "osqp": "osqp>=1,<2",
}
missing = [
    spec for name, spec in packages.items() if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import scipy
from scipy.linalg import solve_discrete_are, expm
from scipy.signal import cont2discrete
import matplotlib
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path
import tempfile
import urllib.request
from matplotlib import font_manager

# Colab과 로컬에서 동일하게 한글 그래프를 표시하기 위한 공개 글꼴.
font_dir = Path(tempfile.gettempdir()) / "mpc_korean_fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic-Regular.ttf"
if not font_path.exists():
    url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    with urllib.request.urlopen(url, timeout=60) as response:
        font_path.write_bytes(response.read())
font_manager.fontManager.addfont(str(font_path))
korean_font = font_manager.FontProperties(fname=font_path).get_name()

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.8),
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 11,
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": ["DejaVu Sans", korean_font],
        "mathtext.fontset": "dejavusans",
        "axes.unicode_minus": False,
    }
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
CHECKS = {}


def finish_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()


def riccati(A, B, Q, R, Pf, N):
    # 부호 관례: u = K x. K에 음의 부호를 포함한다.
    Ps = [None] * (N + 1)
    Ks = [None] * N
    Ps[N] = Pf.copy()
    for k in range(N - 1, -1, -1):
        Pnext = Ps[k + 1]
        Ks[k] = -np.linalg.solve(R + B.T @ Pnext @ B, B.T @ Pnext @ A)
        Ps[k] = Q + A.T @ Pnext @ A + A.T @ Pnext @ B @ Ks[k]
        Ps[k] = (Ps[k] + Ps[k].T) / 2
    return Ps, Ks


def simulate_linear(A, B, policy, x0, steps):
    xs = [np.array(x0, dtype=float)]
    us = []
    for k in range(steps):
        u = np.atleast_1d(policy(xs[-1], k))
        us.append(u)
        xs.append(A @ xs[-1] + B @ u)
    return np.array(xs), np.array(us)


def checked_solve(problem):
    """실패·비유한 값·큰 제약 잔차가 있으면 이전 해를 반환하지 않는다."""
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, max_iter=20000)
    if problem.status != cp.OPTIMAL or not np.isfinite(problem.value):
        raise RuntimeError(f"최적화 실패: {problem.status}")
    violation = max(float(np.max(con.violation())) for con in problem.constraints)
    if not np.isfinite(violation) or violation > 2e-6:
        raise RuntimeError(f"제약 잔차가 허용오차를 넘었습니다: {violation}")


In [ ]:
from scipy.linalg import block_diag, solve_discrete_lyapunov
from scipy.optimize import minimize_scalar
from scipy.signal import place_poles

CHECKS = {}


def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def spectral(M):
    return float(np.max(abs(np.linalg.eigvals(M))))


def condensed(A, B, Q, R, P, N):
    """비용은 x0…x(N−1), 종단 비용은 x(N)에 부과한다."""
    if isinstance(N, bool) or not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N은 양의 정수여야 합니다.")
    A, B, Q, R, P = map(lambda v: np.asarray(v, dtype=float), (A, B, Q, R, P))
    if B.ndim != 2 or A.shape != (B.shape[0], B.shape[0]):
        raise ValueError("A와 B의 상태 차원이 일치해야 합니다.")
    nx, nu = B.shape
    if Q.shape != (nx, nx) or P.shape != (nx, nx) or R.shape != (nu, nu):
        raise ValueError("비용 행렬 차원이 동역학과 맞지 않습니다.")
    if not all(np.all(np.isfinite(v)) for v in (A, B, Q, R, P)):
        raise ValueError("모델과 비용 행렬은 유한해야 합니다.")
    for mat, positive in [(Q, False), (P, False), (R, True)]:
        if not np.allclose(mat, mat.T, rtol=0, atol=1e-12):
            raise ValueError("비용 행렬은 대칭이어야 합니다.")
        least = np.linalg.eigvalsh(mat).min()
        if (positive and least <= 0) or (not positive and least < -1e-12):
            raise ValueError("Q·P는 양의 준정부호, R은 양의 정부호여야 합니다.")
    n, m = B.shape
    Sx = np.vstack([np.linalg.matrix_power(A, i + 1) for i in range(N)])
    Su = np.zeros((N * n, N * m))
    for i in range(N):
        for j in range(i + 1):
            Su[i * n : (i + 1) * n, j * m : (j + 1) * m] = (
                np.linalg.matrix_power(A, i - j) @ B
            )
    Qb = block_diag(*([Q] * (N - 1) + [P]))
    Rb = np.kron(np.eye(N), R)
    H = Su.T @ Qb @ Su + Rb
    F = Su.T @ Qb @ Sx
    Y = Q + Sx.T @ Qb @ Sx
    return H, F, Y, Sx, Su, Qb, Rb


def qcost(U, H, f, constant=0.0):
    return float(0.5 * U @ H @ U + f @ U + constant)


def validate_blocks(groups, size):
    if not groups:
        raise ValueError("참여자 블록이 필요합니다.")
    arrays = [np.asarray(idx) for idx in groups]
    if any(
        idx.ndim != 1 or not len(idx) or not np.issubdtype(idx.dtype, np.integer)
        for idx in arrays
    ):
        raise ValueError("블록은 비어 있지 않은 정수 인덱스 열이어야 합니다.")
    if not np.array_equal(np.sort(np.concatenate(arrays)), np.arange(size)):
        raise ValueError("각 입력은 정확히 하나의 참여자 블록에 속해야 합니다.")


def cooperative_step(U, H, f, groups):
    """같은 이전 계획에서 만든 전체 후보들의 균등 평균을 반환한다."""
    U, H, f = map(lambda v: np.asarray(v, dtype=float), (U, H, f))
    if U.ndim != 1 or H.shape != (U.size, U.size) or f.shape != U.shape:
        raise ValueError("입력 열과 이차 비용의 차원이 맞지 않습니다.")
    if not all(np.all(np.isfinite(v)) for v in (U, H, f)) or not np.allclose(H, H.T):
        raise ValueError("유한 벡터와 대칭 비용 행렬이 필요합니다.")
    try:
        np.linalg.cholesky(H)
    except np.linalg.LinAlgError as error:
        raise ValueError("비용 행렬 H는 양의 정부호여야 합니다.") from error
    validate_blocks(groups, U.size)
    candidates = []
    for idx in groups:
        z = U.copy()
        z[idx] -= np.linalg.solve(H[np.ix_(idx, idx)], (H @ U + f)[idx])
        candidates.append(z)
    out = np.mean(candidates, axis=0)
    assert qcost(out, H, f) <= qcost(U, H, f) + 1e-9
    return out


A = np.diag([0.75, 0.65])
B = np.array([[1.0, 0.4], [0.3, 1.0]])
Q = np.eye(2)
R = 0.2 * np.eye(2)
P = solve_discrete_lyapunov(A.T, Q)
N = 6
H, F, Y, Sx, Su, Qb, Rb = condensed(A, B, Q, R, P, N)
groups = [np.arange(i, N * 2, 2) for i in range(2)]
assert np.linalg.eigvalsh(H).min() > 0 and np.max(abs(A.T @ P @ A - P + Q)) < 1e-12


## 6.1 Introduction and Preliminary Results — 개요와 준비 결과

**교재 인쇄 pp.369–370**

공정 전체의 제어 문제를 여러 참여자의 입력 결정 문제로 나눕니다. 이 장의 분권 제어는 다른 참여자의 계획을 받지 않고, 비협조·협조 제어는 다른 입력 계획을 공유합니다. 두 방식의 핵심 차이는 각자의 비용을 줄이는지, 같은 전체 비용을 줄이는지입니다. 중앙집중 최적해는 성능 비교의 기준입니다.

### 6.1.1 Least Squares Solution — 최소제곱과 선형 MPC의 해

**교재 인쇄 pp.370–375**

예측 상태 열을 $X=S_xx_0+S_uU$로 나타내면 선형 MPC는 입력 열만을 변수로 하는 이차 문제가 됩니다. 상태를 제거하면 작은 변수 집합을 얻지만 조밀한 행렬이 만들어질 수 있습니다. 상태와 입력을 함께 유지한 KKT 시스템과 동적 계획법의 리카티 재귀도 같은 문제를 풉니다.

보충 모델은 안정한 2상태·2입력 시스템입니다. 영 입력의 무한 구간 비용을 종단 비용으로 사용하므로 $A^TPA-P=-Q$입니다. 세 해를 같은 초기 상태·예측 구간·가중치로 비교합니다. 실행 시간 우열이나 불안정 시스템의 조건수를 이 작은 예로 일반화하지 않습니다.

In [ ]:
x0 = np.array([1.0, -0.6])
Uc = -np.linalg.solve(H, F @ x0)
# Explicit state variables X, followed by U; E[X;U]=rhs.
n = 2
m = 2
E = np.zeros((N * n, N * n + N * m))
rhs = np.zeros(N * n)
for k in range(N):
    E[k * n : (k + 1) * n, k * n : (k + 1) * n] = np.eye(n)
    if k == 0:
        rhs[:n] = A @ x0
    else:
        E[k * n : (k + 1) * n, (k - 1) * n : k * n] = -A
    E[k * n : (k + 1) * n, N * n + k * m : N * n + (k + 1) * m] = -B
D = block_diag(Qb, Rb)
M = np.block([[D, E.T], [E, np.zeros((N * n, N * n))]])
sol = np.linalg.solve(M, np.r_[np.zeros(D.shape[0]), rhs])
Uk = sol[N * n : N * n + N * m]
Ps, Ks = riccati(A, B, Q, R, P, N)
xx = x0.copy()
Ur = []
for Kt in Ks:
    uu = Kt @ xx
    Ur.extend(uu)
    xx = A @ xx + B @ uu
Ur = np.array(Ur)
gap = max(np.max(abs(Uc - Uk)), np.max(abs(Uc - Ur)))
assert gap < 1e-10 and np.max(abs(E @ sol[: D.shape[0]] - rhs)) < 1e-10
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for i, ax in enumerate(axs):
    ax.plot(Uc.reshape(N, 2)[:, i], label="상태 제거")
    ax.plot(Uk.reshape(N, 2)[:, i], "o", label="KKT")
    ax.plot(Ur.reshape(N, 2)[:, i], "x", label="리카티")
    ax.set(xlabel="예측 단계", ylabel=f"참여자 {i+1}의 입력")
    ax.legend()
CHECKS["LQ 세 해의 최대 차이"] = gap
CHECKS["KKT 동역학 최대 잔차"] = np.max(abs(E @ sol[: D.shape[0]] - rhs))
savefig(fig, "lq_solutions")


### 6.1.2 Stability of Suboptimal MPC — 유한 반복 MPC의 안정성

**교재 인쇄 pp.375–380**

매 시간 최적화를 끝까지 풀지 않으면 이전 입력 열이 제어기의 기억으로 남습니다. 현재 상태만 보아서는 같은 상태에서 같은 입력이 나온다고 말할 수 없습니다. 교재는 상태와 입력 열을 함께 고려하고, 상태 근처에서 입력 열의 크기도 제한하는 조건을 논의합니다.

보충 예에서는 안정한 모델, 영 입력 종단 비용, 입력 상자 제약을 사용합니다. 이전 입력 열의 첫 항을 버리고 끝에 0을 붙여 다음 시작점을 만듭니다. 매 시간 투영 기울기 한 번만 수행하되 비용이 감소하는 보폭을 사용합니다. 종단 등식 덕분에 이동 시작점 비용은 단계 비용만큼 감소합니다. 아래는 비용 감소와 경로 수렴의 검사이며, $x$만의 균일 리아푸노프 안정성을 자동으로 주장하지 않습니다.

In [ ]:
xx = x0.copy()
warm = np.zeros(2 * N)
xs = [xx.copy()]
values = []
stage = []
step_res = []
alpha = 1 / np.linalg.eigvalsh(H).max()
for k in range(45):
    f = F @ xx
    U = np.clip(warm - alpha * (H @ warm + f), -0.45, 0.45)
    assert qcost(U, H, f) <= qcost(warm, H, f) + 1e-10
    value = qcost(U, H, f, 0.5 * xx @ Y @ xx)
    uu = U[:2]
    ell = 0.5 * (xx @ Q @ xx + uu @ R @ uu)
    xn = A @ xx + B @ uu
    wn = np.r_[U[2:], 0.0, 0.0]
    residual = qcost(wn, H, F @ xn, 0.5 * xn @ Y @ xn) - value + ell
    assert abs(residual) < 1e-10
    values.append(value)
    stage.append(ell)
    step_res.append(residual)
    warm = wn
    xx = xn
    xs.append(xx.copy())
xs = np.array(xs)
values = np.array(values)
stage = np.array(stage)
assert np.max(np.diff(values) + stage[:-1]) < 1e-10 and np.linalg.norm(xs[-1]) < 1e-4
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(xs[:, 0], label="상태 1")
axs[0].plot(xs[:, 1], label="상태 2")
axs[0].set(xlabel="시간 단계", ylabel="상태")
axs[0].legend()
axs[1].semilogy(np.maximum(values, 1e-16))
axs[1].set(xlabel="시간 단계", ylabel="입력 열을 포함한 비용")
CHECKS["이동 시작점 비용 등식 최대 잔차"] = max(abs(np.array(step_res)))
CHECKS["유한 반복 제어의 최종 상태 크기"] = np.linalg.norm(xs[-1])
savefig(fig, "suboptimal")


#### 보충 실습: 같은 상태에서도 이전 입력 계획이 영향을 준다

원본 §6.1.2가 제어기 상태에 입력 열을 포함하는 이유를 직접 확인합니다. **두 경우 모두 물리 상태를 원점에서 시작**하고 이전 계획만 0 또는 모든 성분 0.3으로 둡니다. 각 시간에 투영 기울기 한 번만 수행합니다.

입력 계획이 0이면 원점을 유지하지만, 원점과 맞지 않는 계획은 비용을 낮추는 갱신 후에도 0이 아닐 수 있어 상태가 원점에서 움직입니다. 이는 기존 실습의 전체 비용 감소와 모순되지 않습니다. 증명에 필요한 $\|U\|\le d\|x\|$ 같은 추가 조건을 생략한 **보충 반례**입니다. 원점에서 이 조건은 $U=0$을 요구합니다. 수렴 경로 하나만으로 물리 상태의 원점 불변성이나 균일 안정성을 결론짓지 않습니다.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
first = []
for memory, label in [(0.0, "초기 계획 0"), (0.3, "초기 계획 0.3")]:
    state = np.zeros(2)
    plan = np.full(2 * N, memory)
    path = [state.copy()]
    cost_trace = []
    for k in range(35):
        f = F @ state
        new = np.clip(plan - alpha * (H @ plan + f), -0.45, 0.45)
        assert qcost(new, H, f) <= qcost(plan, H, f) + 1e-10
        if k == 0:
            first.append(np.linalg.norm(new[:2]))
        cost_trace.append(qcost(new, H, f, 0.5 * state @ Y @ state))
        state = A @ state + B @ new[:2]
        plan = np.r_[new[2:], 0.0, 0.0]
        path.append(state.copy())
    axs[0].plot(np.linalg.norm(path, axis=1), label=label)
    axs[1].plot(cost_trace, label=label)
assert first[0] == 0 and first[1] > 0.01
axs[0].set(xlabel="시간 단계 k", ylabel="물리 상태 크기")
axs[1].set(xlabel="시간 단계 k", ylabel="상태와 입력 계획의 비용")
for ax in axs:
    ax.legend()
CHECKS["원점에서 비영 기억이 만드는 첫 입력 크기"] = first[1]
savefig(fig, "controller_memory")


## 6.2 Unconstrained Two-Player Game — 무제약 두 참여자 게임

**교재 인쇄 pp.380–382**

한 참여자의 입력은 자신의 출력뿐 아니라 다른 출력에도 영향을 줍니다. 다른 입력의 영향을 모델에서 버리는 분권 제어와, 그 영향을 알면서 자기 비용만 줄이는 비협조 게임은 실패 원인이 다릅니다.

이 절의 비교용 보충 게임은 $y^+=0.75x+B_gu$, $B_g=[1\;1.8;\;1.8\;1]$인 **한 단계 문제**입니다. 각 비용은 $J_i=\frac12((y_i^+)^2+0.1u_i^2)$입니다. 전체 비용 $J=J_1+J_2$를 기준으로 비교합니다. 이 한 단계 예를 교재의 모든 안정성 조건을 갖춘 일반 MPC로 해석하지 않습니다.

### 6.2.1 Centralized Control — 중앙집중 제어

**교재 인쇄 pp.382–383**

두 입력을 동시에 선택하여 전체 비용을 최소화합니다. 무제약 이차 문제의 해는 선형 방정식으로 구할 수 있습니다. 협조 반복이 충분히 수렴했는지 볼 때도 같은 목적함수의 중앙집중 해를 기준으로 사용합니다. 한 단계 비교용 문제의 중앙집중 폐루프 고유값도 별도로 계산합니다.

### 6.2.2 Decentralized Control — 분권 제어

**교재 인쇄 pp.383–384**

분권 제어는 다른 참여자의 입력 계획 없이 대각 입력–출력 모델로 자기 입력을 설계합니다. 각 국소 폐루프가 안정해도 실제 결합 모델에 그 입력을 동시에 적용한 전체 계는 불안정할 수 있습니다.

보충 실습은 결합 크기를 바꾸며 전체 폐루프의 스펙트럴 반지름을 확인합니다. 초기 상태가 불안정 모드와 정확히 직교한 특별한 경로만 보고 안정성을 판단하지 않도록 고유값을 직접 검사합니다.

In [ ]:
Ag = 0.75 * np.eye(2)
rg = 0.1
Bg = np.array([[1.0, 1.8], [1.8, 1.0]])
Kd = -0.75 / (1 + rg) * np.eye(2)
Hg = Bg.T @ Bg + rg * np.eye(2)
xg = np.array([1.0, -0.5])
g = Bg.T @ Ag @ xg
Kc = -np.linalg.solve(Hg, Bg.T @ Ag)
uc = Kc @ xg
coupling = np.linspace(0, 2, 101)
rd = []
rc = []
for cc in coupling:
    bb = np.array([[1.0, cc], [cc, 1.0]])
    kk = -np.linalg.solve(bb.T @ bb + rg * np.eye(2), bb.T @ Ag)
    rd.append(spectral(Ag + bb @ Kd))
    rc.append(spectral(Ag + bb @ kk))
assert spectral(Ag + Bg @ Kd) > 1 and spectral(Ag + Bg @ Kc) < 1
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(coupling, rd, label="분권 제어")
ax.plot(coupling, rc, label="중앙집중 제어")
ax.axhline(1, color="black", ls="--", label="안정 경계")
ax.set(xlabel="다른 입력의 결합 계수", ylabel="폐루프 스펙트럴 반지름")
ax.legend()
CHECKS["분권 폐루프 스펙트럴 반지름"] = spectral(Ag + Bg @ Kd)
CHECKS["중앙집중 폐루프 스펙트럴 반지름"] = spectral(Ag + Bg @ Kc)
savefig(fig, "decentralized")


### 6.2.3 Noncooperative Game — 비협조 게임

**교재 인쇄 pp.384–392**

비협조 참여자는 상대의 입력 계획을 정확히 알면서 자신의 비용만 최소화합니다. 자신의 입력이 다른 출력에 주는 영향은 목적함수에 포함하지 않습니다. 따라서 모델을 정확히 공유해도 전체 비용이 나쁠 수 있습니다.

보충 게임의 내시점은 각자의 입력 미분이 동시에 0인 점입니다. 이 점의 존재, 최선 반응 반복의 수렴, 실제 폐루프의 안정성은 서로 다른 성질입니다. 동시 최선 반응의 오차 전이행렬을 구해 반복이 발산함을 확인하고, 별도로 내시점을 연립방정식으로 구합니다.

In [ ]:
Mn = Bg + rg * np.eye(2)
un = -np.linalg.solve(Mn, Ag @ xg)
Tbr = -1.8 / (1 + rg) * np.array([[0.0, 1.0], [1.0, 0.0]])
assert np.max(abs(Ag @ xg + Mn @ un)) < 1e-12 and spectral(Tbr) > 1
seq = [np.zeros(2)]
for p in range(12):
    seq.append(-(Ag @ xg + np.array([[0.0, 1.8], [1.8, 0.0]]) @ seq[-1]) / (1 + rg))
seq = np.array(seq)
whole = lambda u: 0.5 * (np.linalg.norm(Ag @ xg + Bg @ u) ** 2 + rg * (u @ u))
assert whole(un) > whole(uc) + 1e-5
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].semilogy(np.linalg.norm(seq - un, axis=1))
axs[0].set(xlabel="최선 반응 반복 p", ylabel="내시점까지의 거리")
axs[1].bar(["중앙집중 최적해", "연립식으로 구한 내시점"], [whole(uc), whole(un)])
axs[1].set(ylabel="전체 비용")
CHECKS["비협조 최선 반응 반복의 스펙트럴 반지름"] = spectral(Tbr)
CHECKS["내시점의 전체 비용 손실"] = whole(un) - whole(uc)
savefig(fig, "noncooperative")


#### 원본 예제 6.9–6.11: 반복 수렴과 폐루프 안정성의 분리

**원본 인쇄 pp.389–392의 모델과 설계값을 재현**합니다. 네 전달함수를 각각 상태공간으로 바꾸고 표본 주기 0.2의 영차 유지로 이산화합니다. 전체 상태는 7개, 예측 구간은 30, 출력 가중치는 1, 입력 가중치는 0.01, 종단 비용은 0, 후보 가중치는 각 0.5입니다.

예제 6.9의 전달함수는
$$G(s)=\begin{bmatrix}
\frac{1}{s^2+0.4s+1}&\frac{0.5}{0.225s+1}\\
\frac{-0.5}{(0.5s+1)(0.25s+1)}&\frac{1.5}{0.75s^2+1.2s+1}
\end{bmatrix}.$$
예제 6.10은 출력 행을 교환합니다. 예제 6.11은 6.9의 비대각 경로를 각각 $0.5/(0.9s+1)$, $-0.5/((2s+1)(s+1))$로 느리게 바꿉니다. 정상상태 이득이 같아도 동적 결합은 달라집니다.

국소 최적성 식을 쌓아 $MU+Fx=0$을 구성합니다. $D$를 $M$의 참여자별 대각 블록으로 두면 가중 최선 반응은 $L=I-\frac12D^{-1}M$입니다. **최선 반응을 전부 붙이는 반복에 가중치 0.5를 반영해야 원본 수치와 일치합니다.** 입력 열은 시간 우선 순서이고 참여자별 블록은 홀수/짝수 위치로 나눕니다.

원본은 6.9에서 반복이 불안정함을 설명합니다. 여기서 *unstable Nash equilibrium*은 내시점으로의 반복 안정성을 뜻합니다. 연립식의 해 자체가 없다는 뜻은 아닙니다. 아래 표의 6.9 폐루프 값은 **연립식을 직접 풀어 얻는 가상적인 수렴 해**의 값이며, 발산하는 분산 반복으로 얻는 제어 성능이 아닙니다. 6.10은 반복이 수렴해도 실제 폐루프가 불안정한 핵심 반례입니다.

In [ ]:
from scipy.signal import tf2ss


def textbook_plant(slow=False, swap=False):
    transfers = [
        ([1], [1, 0.4, 1]),
        ([0.5], [0.9 if slow else 0.225, 1]),
        ([-0.5], [2, 3, 1] if slow else [0.125, 0.75, 1]),
        ([1.5], [0.75, 1.2, 1]),
    ]
    aa = []
    bb = []
    cc = []
    for num, den in transfers:
        a, b, c, d = tf2ss(num, den)
        a, b, c, d, _ = cont2discrete((a, b, c, d), 0.2)
        aa.append(a)
        bb.append(b)
        cc.append(c)
    A = block_diag(*aa)
    B = np.zeros((7, 2))
    C = np.zeros((2, 7))
    pos = 0
    for j in range(4):
        n = aa[j].shape[0]
        B[pos : pos + n, j % 2] = bb[j][:, 0]
        C[j // 2, pos : pos + n] = cc[j][0]
        pos += n
    return A, B, C[::-1] if swap else C


def textbook_game(A, B, C):
    N = 30
    n = 7
    Sx = np.vstack([np.linalg.matrix_power(A, k) for k in range(N)])
    Su = np.zeros((N * n, N * 2))
    for k in range(N):
        for j in range(k):
            Su[k * n : (k + 1) * n, j * 2 : (j + 1) * 2] = (
                np.linalg.matrix_power(A, k - 1 - j) @ B
            )
    M = np.zeros((60, 60))
    f = np.zeros((60, 7))
    scaled = np.zeros((60, 60))
    for i in range(2):
        idx = np.arange(i, 60, 2)
        Cy = np.kron(np.eye(N), C[i : i + 1])
        V = Cy @ Su
        W = Cy @ Sx
        M[idx, :] = V[:, idx].T @ V
        M[np.ix_(idx, idx)] += 0.01 * np.eye(N)
        f[idx, :] = V[:, idx].T @ W
        scaled[idx, :] = np.linalg.solve(M[np.ix_(idx, idx)], M[idx, :])
    L = np.eye(60) - 0.5 * scaled
    plan_gain = -np.linalg.solve(M, f)
    K = plan_gain[:2, :]
    assert np.linalg.norm(M @ plan_gain + f) < 1e-9
    return spectral(L), spectral(A + B @ K), L, K


textbook_results = []
textbook_models = []
for number, slow, swapped, expected in [
    (9, False, False, (1.11, None)),
    (10, False, True, (0.63, 1.03)),
    (11, True, False, (0.61, 0.88)),
]:
    aa, bb, cc = textbook_plant(slow, swapped)
    rr, rc, ll, kk = textbook_game(aa, bb, cc)
    assert abs(rr - expected[0]) < 0.005
    if expected[1] is not None:
        assert abs(rc - expected[1]) < 0.005
    textbook_results.append((rr, rc))
    textbook_models.append((aa, bb, cc))
    CHECKS[f"원본 예제 6.{number} 반복 스펙트럴 반지름"] = rr
    CHECKS[f"원본 예제 6.{number} 연립해 폐루프 스펙트럴 반지름"] = rc
values = np.array(textbook_results)
ii = np.arange(3)
fig, ax = plt.subplots(figsize=(10, 3.7))
ax.bar(ii - 0.18, values[:, 0], 0.36, label="분산 반복 L")
ax.bar(ii + 0.18, values[:, 1], 0.36, label="연립해를 적용한 폐루프")
ax.axhline(1, color="black", ls="--", label="안정 경계")
for j in range(3):
    ax.text(
        j - 0.18, values[j, 0] + 0.02, f"{values[j,0]:.3f}", ha="center", fontsize=10
    )
    ax.text(
        j + 0.18, values[j, 1] + 0.02, f"{values[j,1]:.3f}", ha="center", fontsize=10
    )
ax.set(
    xticks=ii,
    xticklabels=["예제 6.9\n폐루프 값은 가상 연립해", "예제 6.10", "예제 6.11"],
    ylabel="스펙트럴 반지름",
    ylim=(0, 1.4),
)
ax.legend(ncol=3, fontsize=9)
savefig(fig, "textbook_stability")
# The same DC gain can conceal radically different transient interactions.
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
dc = []
for (aa, bb, cc), label in [
    (textbook_models[0], "예제 6.9"),
    (textbook_models[2], "예제 6.11"),
]:
    dc.append(cc @ np.linalg.solve(np.eye(7) - aa, bb))
    state = np.zeros(7)
    ys = []
    for k in range(101):
        ys.append(cc @ state)
        state = aa @ state + bb[:, 1]
    ys = np.array(ys)
    axs[0].plot(0.2 * np.arange(101), ys[:, 0], label=label)
    state = np.zeros(7)
    ys = []
    for k in range(101):
        ys.append(cc @ state)
        state = aa @ state + bb[:, 0]
    axs[1].plot(0.2 * np.arange(101), np.array(ys)[:, 1], label=label)
assert np.max(abs(dc[0] - dc[1])) < 1e-12
for ax, title in zip(axs, ["입력 2 → 출력 1", "입력 1 → 출력 2"]):
    ax.set(title=title, xlabel="시간 [s]", ylabel="단위 계단 응답")
    ax.legend()
CHECKS["원본 6.9와 6.11 정상상태 이득 최대 차이"] = np.max(abs(dc[0] - dc[1]))
savefig(fig, "textbook_coupling")


### 6.2.4 Cooperative Game — 협조 게임

**교재 인쇄 pp.392–398**

각 참여자는 자기 입력만 바꾸지만 공통의 전체 비용을 최소화합니다. 참여자 1의 후보는 $(u_1^*,u_2^p)$, 참여자 2의 후보는 $(u_1^p,u_2^*)$입니다. **각자 새 입력을 모두 붙이는 것**과 **전체 후보 벡터들을 평균하는 것**은 다릅니다.

볼록 목적함수에서 후보들의 볼록 결합은 가중 후보 비용보다 크지 않습니다. 보충 예는 두 후보를 절반씩 합칩니다. 양의 정부호 이차 비용이므로 유일한 중앙집중 해와의 거리와 비용 차이를 확인합니다.

In [ ]:
seqc = [np.zeros(2)]
for p in range(300):
    seqc.append(cooperative_step(seqc[-1], Hg, g, [np.array([0]), np.array([1])]))
seqc = np.array(seqc)
cc = np.array([whole(u) for u in seqc])
assert np.diff(cc).max() < 1e-12 and np.linalg.norm(seqc[-1] - uc) < 1e-9
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(cc)
axs[0].axhline(whole(uc), color="orange", ls="--", label="중앙집중 최솟값")
axs[0].set(xlabel="협조 반복 p", ylabel="전체 비용")
axs[0].legend()
axs[1].semilogy(np.maximum(np.linalg.norm(seqc - uc, axis=1), 1e-16))
axs[1].set(xlabel="협조 반복 p", ylabel="중앙집중 해까지의 거리")
CHECKS["협조 해와 중앙집중 해의 최대 차이"] = np.max(abs(seqc[-1] - uc))
CHECKS["협조 반복 비용 증가 최댓값"] = np.diff(cc).max()
savefig(fig, "cooperative")


### 6.2.5 Tracking Nonzero Setpoints — 0이 아닌 설정값 추적

**교재 인쇄 pp.398–405**

설정값 추적에는 평형 입력·출력을 먼저 계산합니다. 정상상태 이득 $G=C(I-A)^{-1}B$에도 참여자 간 결합이 남습니다. 대각 성분만으로 각자의 목표 입력을 구하면 전체 출력은 설정값과 달라질 수 있습니다.

보충 실습은 이전 게임 모델의 정상상태 목표 문제 $\min\frac12\|Gu-r\|^2$를 협조 반복으로 풉니다. 이는 **목표 계산**의 반복이며 동적 MPC 폐루프의 수렴과 구분합니다.

In [ ]:
Gss = np.linalg.solve(np.eye(2) - Ag, Bg)
ref = np.array([1.0, -0.5])
ut = np.linalg.solve(Gss, ref)
ud = ref / np.diag(Gss)
Ht = Gss.T @ Gss
gt = -Gss.T @ ref
uu = np.zeros(2)
trace = []
for p in range(400):
    uu = cooperative_step(uu, Ht, gt, [np.array([0]), np.array([1])])
    trace.append(np.linalg.norm(Gss @ uu - ref))
assert np.linalg.norm(uu - ut) < 1e-9
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
ii = np.arange(2)
axs[0].bar(ii - 0.2, Gss @ ud, 0.4, label="분권 목표의 실제 출력")
axs[0].bar(ii + 0.2, Gss @ uu, 0.4, label="협조 목표의 실제 출력")
axs[0].plot(ii, ref, "kx", label="설정값")
axs[0].set(xticks=ii, xticklabels=["출력 1", "출력 2"], ylabel="정상상태 출력")
axs[0].legend(fontsize=9)
axs[1].semilogy(np.maximum(trace, 1e-16))
axs[1].set(xlabel="목표 계산 반복 p", ylabel="출력 목표 잔차")
CHECKS["분권 목표의 출력 잔차"] = np.linalg.norm(Gss @ ud - ref)
CHECKS["협조 목표의 출력 잔차"] = np.linalg.norm(Gss @ uu - ref)
savefig(fig, "targets")


#### 원본 예제 6.12: 출력 짝짓기와 목표 계산

**원본 인쇄 pp.401–404**의 $G=[-0.5,1;2,1]$, 설정값 $(1,1)$을 사용합니다. 분권 목표 $u_i=r_i/G_{ii}$를 시작점으로 삼고 후보 가중치 0.5로 10회 반복합니다. 두 번째 경우는 출력 행만 교환합니다.

비협조 목표 갱신은 $u_i^*=(r_i-G_{ij}u_j)/G_{ii}$이고, 협조 목표 갱신은 전체 비용 $\frac12\|Gu-r\|^2$를 자기 입력에 대해 최소화합니다. 두 경우 모두 **전체 후보의 평균**을 계산합니다. 원본의 반복 스펙트럴 반지름은 비협조 1.12/0.559, 협조 0.757/0.757입니다. 같은 설정값과 동일 출력 가중치에서는 행 교환으로 $G^TG$가 변하지 않기 때문에 협조 반복 행렬도 같습니다.

그래프는 첫 10회의 출력 목표 잔차입니다. 안정한 반복도 10회 만에 정확히 수렴하는 것은 아니므로, 별도로 150회 후 중앙집중 목표와의 일치를 검사합니다. 목표 계산의 반복을 실제 시스템의 시간 경로로 해석하지 않습니다.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.7))
co_matrices = []
for case, gg in enumerate(
    [np.array([[-0.5, 1.0], [2.0, 1.0]]), np.array([[2.0, 1.0], [-0.5, 1.0]])]
):
    target = np.ones(2)
    central = np.linalg.solve(gg, target)
    decentral = target / np.diag(gg)
    hh = gg.T @ gg
    ff = -gg.T @ target
    nc_matrix = np.eye(2) - 0.5 * np.diag(1 / np.diag(gg)) @ gg
    co_matrix = np.eye(2) - 0.5 * np.diag(1 / np.diag(hh)) @ hh
    co_matrices.append(co_matrix)
    nc = decentral.copy()
    co = decentral.copy()
    nerr = [np.linalg.norm(gg @ nc - target)]
    cerr = nerr.copy()
    for _ in range(150):
        nc = nc_matrix @ nc + 0.5 * target / np.diag(gg)
        co = cooperative_step(co, hh, ff, [np.array([0]), np.array([1])])
        nerr.append(np.linalg.norm(gg @ nc - target))
        cerr.append(np.linalg.norm(gg @ co - target))
    assert np.linalg.norm(co - central) < 1e-10
    if case == 1:
        assert np.linalg.norm(nc - central) < 1e-10
    expected_nc = [np.sqrt(1.25), np.sqrt(0.3125)][case]
    assert abs(spectral(nc_matrix) - expected_nc) < 1e-12
    axs[case].semilogy(range(11), nerr[:11], "o-", label="비협조 목표 반복")
    axs[case].semilogy(range(11), cerr[:11], "o-", label="협조 목표 반복")
    axs[case].set(
        title="원래 출력 순서" if case == 0 else "출력 행 교환",
        xlabel="목표 계산 반복 p",
        ylabel="출력 목표 잔차",
    )
    axs[case].legend(fontsize=9)
    CHECKS[f"원본 6.12 경우 {case+1} 비협조 반복 반지름"] = spectral(nc_matrix)
    CHECKS[f"원본 6.12 경우 {case+1} 협조 반복 반지름"] = spectral(co_matrix)
assert np.max(abs(co_matrices[0] - co_matrices[1])) < 1e-12
savefig(fig, "textbook_targets")


### 6.2.6 State Estimation — 상태 추정

**교재 인쇄 pp.405–406**

참여자는 국소 측정을 사용해 자기 상태를 추정하지만, 모델에 들어가는 다른 참여자의 **실제 적용 입력**도 알아야 합니다. 알려진 입력을 모두 반영하면 결정론적 추정 오차계가 관측기 오차 동역학만으로 분리됩니다.

보충 예는 과정·측정 잡음 없이 예측형 관측기를 비교합니다. 각 시간의 입력을 적용한 뒤 현재 측정으로 다음 추정을 갱신합니다. 다른 입력을 누락하면 지속적인 모델 오차가 생깁니다. 교재의 잡음 없는 오차 수렴 결과를 잡음이 지속되는 상황에 그대로 적용하지 않습니다.

In [ ]:
xx = np.array([1.0, -0.5])
good = np.zeros(2)
bad = np.zeros(2)
ua = np.array([0.2, -0.1])
eg = []
eb = []
for k in range(45):
    eg.append(np.linalg.norm(xx - good))
    eb.append(np.linalg.norm(xx - bad))
    gn = Ag @ good + Bg @ ua + 0.4 * (xx - good)
    bn = Ag @ bad + np.diag(np.diag(Bg)) @ ua + 0.4 * (xx - bad)
    xx = Ag @ xx + Bg @ ua
    good, bad = gn, bn
assert eg[-1] < 1e-12 and eb[-1] > 0.1
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.semilogy(np.maximum(eg, 1e-16), label="두 실제 입력을 반영")
ax.semilogy(eb, label="상대 입력을 누락")
ax.set(xlabel="시간 단계", ylabel="상태 추정 오차 크기")
ax.legend()
CHECKS["입력 공유 관측기의 마지막 오차"] = eg[-1]
CHECKS["상대 입력 누락 관측기의 마지막 오차"] = eb[-1]
savefig(fig, "local_estimation")


## 6.3 Constrained Two-Player Game — 제약이 있는 두 참여자 게임

**교재 인쇄 pp.406–408**

이제 협조 제어에 집중합니다. 독립된 입력 제약과 참여자가 함께 쓰는 자원 제약은 성질이 다릅니다. 교재는 불안정 모드의 종단 제거와 원점 근처의 입력 열 크기 제한 등 안정성 조건도 추가합니다.

여기서는 §6.1의 안정한 보충 모델에 상자 입력 제약을 적용해 **고정 상태에서의 최적화 반복**을 확인합니다. 불안정 계의 종단 모드 제거나 교재 정리의 모든 안정성 제약을 구현하는 실습은 아닙니다. 상태 강제약과 출력 연성 제약도 추가하지 않습니다.

### 6.3.1 Uncoupled Input Constraints — 독립 입력 제약

**교재 인쇄 pp.408–411**

허용 집합이 참여자별 곱집합이면, 한 참여자의 입력을 바꿀 때 다른 참여자의 허용 범위가 바뀌지 않습니다. 각 후보가 실행 가능하고 전체 비용을 줄이면 후보들의 볼록 결합도 실행 가능하며 비용을 증가시키지 않습니다.

보충 실습은 각 입력의 절댓값을 0.4 이하로 제한한 예측 구간 6의 QP입니다. 각 참여자는 같은 이전 입력 열을 기준으로 자기 블록을 정확히 최적화합니다. 고정된 상태의 중앙집중 QP 해와 수렴한 분산 해를 비교합니다.

In [ ]:
xb = np.array([2.0, -1.0])
fb = F @ xb
lim = 0.4
uv = cp.Variable(2 * N)
problem = cp.Problem(
    cp.Minimize(0.5 * cp.quad_form(uv, H) + fb @ uv), [cp.abs(uv) <= lim]
)
checked_solve(problem)
Ustar = uv.value.copy()
locals = []
for idx in groups:
    zz = cp.Variable(N)
    qpar = cp.Parameter(N)
    hh = H[np.ix_(idx, idx)]
    pr = cp.Problem(
        cp.Minimize(0.5 * cp.quad_form(zz, hh) + qpar @ zz), [cp.abs(zz) <= lim]
    )
    locals.append((idx, zz, qpar, pr, hh))
U = np.zeros(2 * N)
costs = [qcost(U, H, fb)]
dist = [np.linalg.norm(U - Ustar)]
maxu = 0.0
for p in range(100):
    cand = []
    for idx, zz, qpar, pr, hh in locals:
        qpar.value = (H @ U + fb)[idx] - hh @ U[idx]
        checked_solve(pr)
        z = U.copy()
        z[idx] = zz.value
        cand.append(z)
    U = np.mean(cand, axis=0)
    assert np.max(abs(U)) <= lim + 2e-7
    maxu = max(maxu, np.max(abs(U)))
    costs.append(qcost(U, H, fb))
    dist.append(np.linalg.norm(U - Ustar))
assert np.max(np.diff(costs)) < 1e-7 and dist[-1] < 2e-5
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].semilogy(np.maximum(dist, 1e-16))
axs[0].set(xlabel="협조 반복 p", ylabel="중앙집중 입력 열까지 거리")
axs[1].plot(U.reshape(N, 2)[:, 0], "o-", label="참여자 1")
axs[1].plot(U.reshape(N, 2)[:, 1], "o-", label="참여자 2")
axs[1].axhline(lim, color="gray", ls="--")
axs[1].axhline(-lim, color="gray", ls="--")
axs[1].set(xlabel="예측 단계", ylabel="수렴한 입력")
axs[1].legend()
CHECKS["상자 제약 분산 해와 중앙집중 해 거리"] = dist[-1]
CHECKS["상자 제약 반복의 입력 최대 크기"] = maxu
savefig(fig, "box_constraints")


#### 보충 실습: 몇 번 반복하면 충분한가 — 계산 가능한 최적성 인증

앞의 독립 입력 상자 QP와 같은 상태·비용을 사용합니다. 중앙집중 해와의 거리는 좋은 검증 지표지만, 실제 분산 계산을 멈출 때 그 해를 이미 안다고 가정할 수 없습니다. 볼록 미분가능 비용 $f$와 실행 가능한 $U$에 대해
$$0\le f(U)-f^*\le g_{\rm FW}(U)=\max_{V\in[-l,l]^n}\nabla f(U)^T(U-V)
=\nabla f(U)^TU+l\|\nabla f(U)\|_1.$$
이는 접평면이 주는 하한에서 얻는 **전체 최적성의 상한**입니다. 블록별 비용 감소량이 작다는 판단과 다릅니다. 공유 결합 제약에서는 상자 공식이 그대로 적용되지 않습니다.

여기서는 $H\succ0$이므로 $\mu=\lambda_{\min}(H)>0$도 이용합니다. $V_\mu=\operatorname{clip}(U-\nabla f(U)/\mu,-l,l)$로 두면 강볼록 하한으로부터
$$f(U)-f^*\le -\nabla f(U)^T(V_\mu-U)-\frac{\mu}{2}\|V_\mu-U\|^2$$
를 얻습니다. 두 인증값 중 작은 값이 $10^{-6}$ 이하이면 멈춥니다. 이는 이 QP의 **절대 목적값 오차** 허용치이며 폐루프 안정성이나 첫 입력의 오차 허용치를 뜻하지 않습니다. 정확한 전체 기울기와 곡률 정보가 필요하므로 통신 비용이 공짜인 기준도 아닙니다.

활성 입력 한계에서는 최적해의 원래 기울기가 0일 필요가 없습니다. 중앙집중 기준은 인증을 검증하는 데만 사용하고 종료 판단에는 사용하지 않습니다. 솔버의 미세한 상자 위반은 후보 투영으로 없애 실행 가능성 전제를 유지합니다.


In [ ]:
def box_gap_certificates(plan, hessian, linear, bound, curvature):
    """실행 가능한 상자 QP의 FW 및 강볼록 목적값 오차 상한."""
    if np.max(abs(plan)) > bound + 1e-12:
        raise ValueError("인증에는 실행 가능한 입력 열이 필요합니다.")
    gradient = hessian @ plan + linear
    fw_gap = gradient @ plan + bound * np.sum(abs(gradient))
    lower_model_step = np.clip(plan - gradient / curvature, -bound, bound) - plan
    strong_gap = -gradient @ lower_model_step - 0.5 * curvature * (lower_model_step @ lower_model_step)
    return max(0.0, float(fw_gap)), max(0.0, float(strong_gap))

mu_box = float(np.linalg.eigvalsh(H)[0])
assert mu_box > 0
certificate_plan = np.zeros(2 * N)
certificate_history = []
objective_tolerance = 1e-6
central_feasible = np.clip(Ustar, -lim, lim)
central_cost = qcost(central_feasible, H, fb)
for iteration in range(201):
    fw_gap, strong_gap = box_gap_certificates(certificate_plan, H, fb, lim, mu_box)
    measured_gap = qcost(certificate_plan, H, fb) - central_cost
    certificate_history.append([measured_gap, fw_gap, strong_gap])
    # 수치 기준 해와 인증을 대조; 기준 해는 종료 조건에 들어가지 않는다.
    assert measured_gap >= -2e-7
    assert measured_gap <= min(fw_gap, strong_gap) + 2e-7
    if min(fw_gap, strong_gap) <= objective_tolerance:
        break
    if iteration == 200:
        raise RuntimeError("정한 반복 예산 안에 최적성 인증을 얻지 못했습니다.")
    candidates = []
    for idx, zz, qpar, local_problem, hh in locals:
        qpar.value = (H @ certificate_plan + fb)[idx] - hh @ certificate_plan[idx]
        checked_solve(local_problem)
        candidate = certificate_plan.copy()
        candidate[idx] = np.clip(zz.value, -lim, lim)
        candidates.append(candidate)
    certificate_plan = np.mean(candidates, axis=0)

certificate_history = np.asarray(certificate_history)
raw_gradient_norm = np.linalg.norm(H @ central_feasible + fb)
projected_gradient_norm = np.linalg.norm(
    central_feasible - np.clip(central_feasible - (H @ central_feasible + fb), -lim, lim)
)
assert raw_gradient_norm > 0.1
assert projected_gradient_norm < 2e-6
assert np.max(abs(certificate_plan)) <= lim + 1e-12
assert np.max(np.diff(certificate_history[:, 0])) < 2e-7
print(f"인증 종료: {iteration}회, 상한={min(fw_gap, strong_gap):.3e}, 실제 비용 차={measured_gap:.3e}")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.7))
for column, label in enumerate(["중앙집중 기준과 비용 차", "볼록 접평면 상한", "강볼록 하한 기반 상한"]):
    axs[0].semilogy(np.maximum(certificate_history[:, column], 1e-16), label=label)
axs[0].axhline(objective_tolerance, color="black", ls="--", label="종료 허용치")
axs[0].set(xlabel="협조 반복 p", ylabel="목적값 오차 / 인증 상한")
axs[0].legend(fontsize=8)
axs[1].bar(["원래 기울기", "투영 잔차"], [raw_gradient_norm, max(projected_gradient_norm, 1e-16)])
axs[1].set(yscale="log", ylabel="잔차 크기", title="활성 제약이 있는 중앙집중 최적해")
CHECKS["상자 QP 인증 종료 반복 수"] = iteration
CHECKS["상자 QP 종료 목적값 오차 상한"] = min(fw_gap, strong_gap)
CHECKS["상자 QP 종료 실제 목적값 차"] = measured_gap
CHECKS["상자 QP 최적해 원래 기울기 크기"] = raw_gradient_norm
CHECKS["상자 QP 최적해 투영 잔차"] = projected_gradient_norm
savefig(fig, "optimality_certificate")


### 6.3.2 Coupled Input Constraints — 결합 입력 제약

**교재 인쇄 pp.411–413**

공유 자원 제약에서는 한 참여자가 더 쓰려면 다른 참여자가 덜 써야 할 수 있습니다. 상대 입력을 고정한 한 블록씩의 개선만으로는 자원을 재배분할 수 없어 중앙집중 해보다 나쁜 점에 멈출 수 있습니다.

보충 예는 $u_1,u_2\ge0$, $u_1+u_2\le1$, $J=(u_1-1)^2+(u_2-2)^2$입니다. 경계점 $(0.8,0.2)$에서 각각 혼자서는 개선할 수 없지만 함께 움직이면 $(0,1)$에서 더 낮은 비용을 얻습니다. 또한 $(0,0)$에서 각자의 최선 입력을 그대로 붙이면 $(1,1)$로 제약을 위반합니다. 전체 후보 $(1,0),(0,1)$의 평균은 실행 가능합니다.

In [ ]:
resource = lambda u: float((u[0] - 1) ** 2 + (u[1] - 2) ** 2)
stuck = np.array([0.8, 0.2])
br = np.array([min(1.0, 1 - stuck[1]), min(2.0, 1 - stuck[0])])
assert np.max(abs(br - stuck)) < 1e-12
rvar = cp.Variable(2)
pr = cp.Problem(
    cp.Minimize(cp.sum_squares(rvar - [1, 2])), [rvar >= 0, cp.sum(rvar) <= 1]
)
checked_solve(pr)
best = rvar.value
assert np.linalg.norm(best - [0, 1]) < 1e-6 and resource(stuck) > resource(best) + 1
unsafe = np.array([1.0, 1.0])
safe = np.array([0.5, 0.5])
assert unsafe.sum() > 1 and safe.sum() <= 1
xx, yy = np.meshgrid(np.linspace(0, 1, 180), np.linspace(0, 1, 180))
zz = (xx - 1) ** 2 + (yy - 2) ** 2
fig, ax = plt.subplots(figsize=(7, 4))
ax.contour(xx, yy, zz, levels=12, alpha=0.55)
ax.fill([0, 1, 0], [0, 0, 1], alpha=0.15, label="공유 자원 허용 집합")
ax.scatter(*stuck, color="orange", label="블록별 개선이 멈춘 점")
ax.scatter(*best, color="green", label="중앙집중 최적점")
ax.plot([stuck[0], best[0]], [stuck[1], best[1]], "k--")
ax.set(xlabel="참여자 1의 입력", ylabel="참여자 2의 입력")
ax.legend(fontsize=9)
CHECKS["공유 자원 경계점의 전체 비용 손실"] = resource(stuck) - resource(best)
CHECKS["새 입력 단순 결합의 공유 자원 위반량"] = unsafe.sum() - 1
savefig(fig, "coupled_resource")


### 6.3.3 Exponential Convergence with Estimate Error — 추정 오차가 있을 때의 지수 수렴

**교재 인쇄 pp.413–415**

상태 추정 오차가 있으면 예측한 다음 상태와 실제 추정기 갱신이 다릅니다. 교재는 입력 열을 포함하는 제어 비용과 추정 오차의 리아푸노프 함수를 합쳐 분석합니다. 비용의 섭동을 제한하는 연속성·립시츠 조건, 재귀적 실행 가능성, 원점 근처의 입력 열 크기 제한 등이 필요합니다.

여기서 추정 오차는 잡음 없이 지수적으로 감소하는 경우입니다. §6.2.6의 오차 경로는 이 구조를 설명하지만 제약 MPC 정리의 전체 증명은 아닙니다. 잡음이 지속되는 경우의 범위 보장은 5장의 유계 오차 접근처럼 별도로 다뤄야 합니다.

### 6.3.4 Disturbance Models and Zero Offset — 외란 모델과 오프셋 제거

**교재 인쇄 pp.415–419**

각 참여자는 적분 외란을 추가한 국소 상태를 추정합니다. 확장 모델의 검출가능성과 실제 입력 공유가 필요하며, 외란 추정값을 반영한 목표 계산도 참여자 간 결합을 고려해야 합니다.

보충 실습은 §6.1의 안정한 모델에 상수 과정 외란을 넣습니다. 두 국소 확장 관측기가 실제 두 입력을 사용합니다. 목표 평형은 전체 선형 등식으로 계산하고, 동적 입력 열은 시간당 협조 반복 3회로 개선합니다. 이전 실제 입력 열을 이동해 다음 시작점으로 사용합니다.

이 예는 **무제약·잡음 없는 외란 보상 원리의 보충 실습**이며, 이 절의 모든 제약·안정성 조건을 구현하지 않습니다. 모델에 외란을 추가하지 않은 경우와 비교합니다.

In [ ]:
refd = np.array([0.5, -0.3])
disturb = np.array([0.1, -0.06])
Les = []
for aa in np.diag(A):
    ae = np.array([[aa, 1.0], [0.0, 1.0]])
    ce = np.array([[1.0, 0.0]])
    assert np.linalg.matrix_rank(np.vstack([ce, ce @ ae])) == 2
    Les.append(place_poles(ae.T, ce.T, [0.3, 0.4]).gain_matrix.T[:, 0])


def offset_run(aug):
    xx = np.zeros(2)
    xh = np.zeros(2)
    dh = np.zeros(2)
    warm = np.zeros(2 * N)
    traj = [xx.copy()]
    estd = [dh.copy()]
    for k in range(75):
        dt = dh if aug else np.zeros(2)
        us = np.linalg.solve(B, (np.eye(2) - A) @ refd - dt)
        U = warm - np.tile(us, N)
        ff = F @ (xh - refd)
        for p in range(3):
            U = cooperative_step(U, H, ff, groups)
        absolute = U + np.tile(us, N)
        ua = absolute[:2]
        innovation = xx - xh
        if aug:
            xhn = (
                A @ xh
                + B @ ua
                + dh
                + np.array([Les[i][0] * innovation[i] for i in range(2)])
            )
            dh = dh + np.array([Les[i][1] * innovation[i] for i in range(2)])
        else:
            xhn = A @ xh + B @ ua + 0.4 * innovation
        xx = A @ xx + B @ ua + disturb
        xh = xhn
        warm = np.r_[absolute[2:], us]
        traj.append(xx.copy())
        estd.append(dh.copy())
    return np.array(traj), np.array(estd)


base, bd = offset_run(False)
comp, cd = offset_run(True)
assert (
    np.linalg.norm(comp[-1] - refd) < 1e-7 and np.linalg.norm(cd[-1] - disturb) < 1e-8
)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
for i, ax in enumerate(axs):
    ax.plot(base[:, i], label="외란 상태 없음")
    ax.plot(comp[:, i], label="국소 외란 추정")
    ax.axhline(refd[i], color="gray", ls="--", label="설정값")
    ax.set(xlabel="시간 단계", ylabel=f"출력 {i+1}")
    ax.legend()
CHECKS["외란 없는 모델의 최종 출력 오프셋 크기"] = np.linalg.norm(base[-1] - refd)
CHECKS["외란 보상 분산 제어의 최종 출력 오프셋 크기"] = np.linalg.norm(comp[-1] - refd)
CHECKS["국소 외란 추정의 최종 오차 크기"] = np.linalg.norm(cd[-1] - disturb)
savefig(fig, "distributed_offset")


## 6.4 Constrained M-Player Game — 제약이 있는 다수 참여자 게임

**교재 인쇄 pp.419–421**

참여자가 여러 명이어도 각자는 자기 입력 블록만 바꾸고 전체 비용을 평가합니다. 다음 입력 열은 전체 후보들의 양의 가중 평균으로 만들며 가중치의 합은 1이어야 합니다. 균등 가중치라면 각 블록의 새 값은 이전 값에서 최선 반응 방향으로 $1/M$만큼 이동합니다.

보충 예는 3참여자의 엄격 볼록 이차 비용과 독립 상자 제약입니다. 국소 최적해는 스칼라 절단으로 정확히 계산됩니다. 중앙집중 QP 해와 비교하며, 실제 여러 컴퓨터의 통신 속도를 측정하는 실습은 아닙니다.

In [ ]:
Hm = np.full((3, 3), 0.8)
np.fill_diagonal(Hm, 1.0)
gm = np.array([-0.8, 0.4, 0.2])
um = cp.Variable(3)
prob = cp.Problem(cp.Minimize(0.5 * cp.quad_form(um, Hm) + gm @ um), [cp.abs(um) <= 1])
checked_solve(prob)
ustar = um.value
u = np.zeros(3)
tr = [u.copy()]
costm = [qcost(u, Hm, gm)]
for p in range(250):
    cand = []
    for i in range(3):
        v = u.copy()
        v[i] = np.clip(u[i] - (Hm @ u + gm)[i] / Hm[i, i], -1, 1)
        cand.append(v)
    u = np.mean(cand, axis=0)
    tr.append(u.copy())
    costm.append(qcost(u, Hm, gm))
tr = np.array(tr)
assert np.max(np.diff(costm)) < 1e-12 and np.linalg.norm(u - ustar) < 1e-6
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(tr)
axs[0].set(xlabel="협조 반복 p", ylabel="참여자별 입력")
axs[0].legend(["참여자 1", "참여자 2", "참여자 3"])
axs[1].semilogy(np.maximum(np.linalg.norm(tr - ustar, axis=1), 1e-16))
axs[1].set(xlabel="협조 반복 p", ylabel="중앙집중 해까지 거리")
CHECKS["3참여자 해와 중앙집중 해의 거리"] = np.linalg.norm(u - ustar)
savefig(fig, "many_players")


#### 보충 실습: 볼록 문제도 새 블록을 그대로 붙이면 발산할 수 있다

§6.4의 $1/M$ 가중치를 확인하기 위해 앞의 3참여자 목적함수에서 **상자 제약을 제거한** 보충 문제를 비교합니다. $H$의 대각은 1, 비대각은 0.8입니다. 고유값은 0.2,0.2,2.6이므로 엄격 볼록입니다.

새 입력 블록을 모두 붙이면 반복 행렬은 $T_{\rm raw}=I-H$이고 스펙트럴 반지름은 1.6입니다. 전체 후보를 평균하면 $T_{\rm avg}=I-H/3$으로 바뀌며 반지름은 약 0.9333입니다. **목적함수의 볼록성만으로 동시 블록 갱신의 수렴은 보장되지 않습니다.** 가중치는 전체 후보의 볼록 결합을 만드는 알고리즘의 일부입니다. 실제 다중 장치 성능을 측정한 실험은 아닙니다.

In [ ]:
raw_matrix = np.eye(3) - Hm
avg_matrix = np.eye(3) - Hm / 3
exact = -np.linalg.solve(Hm, gm)
raw = np.zeros(3)
avg_plan = raw.copy()
raw_err = []
avg_err = []
for _ in range(31):
    raw_err.append(np.linalg.norm(raw - exact))
    avg_err.append(np.linalg.norm(avg_plan - exact))
    raw = raw_matrix @ raw - gm
    avg_plan = cooperative_step(avg_plan, Hm, gm, [np.array([i]) for i in range(3)])
assert spectral(raw_matrix) > 1 and spectral(avg_matrix) < 1
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogy(raw_err, label="새 블록 단순 결합")
ax.semilogy(avg_err, label="전체 후보를 1/3씩 평균")
ax.set(xlabel="최적화 반복 p", ylabel="중앙집중 해까지의 거리")
ax.legend()
CHECKS["3참여자 단순 결합 반복 반지름"] = spectral(raw_matrix)
CHECKS["3참여자 전체 후보 평균 반복 반지름"] = spectral(avg_matrix)
savefig(fig, "parallel_candidates")


## 6.5 Nonlinear Distributed MPC — 비선형 분산 MPC

**교재 인쇄 pp.421**

비선형 물리 모델에서는 다른 부분 시스템의 상태와 입력이 함께 동역학에 들어갈 수 있습니다. 각 참여자가 자기 입력만 바꾸더라도 전체 상태 경로와 비용을 평가해야 합니다. 볼록 선형 문제에서 사용한 평균 후보의 비용 감소는 비선형 비용에서 자동으로 성립하지 않습니다.

### 6.5.1 Nonconvexity — 비볼록성

**교재 인쇄 pp.421–423**

교재 그림 6.7의 목적함수에 상수 2를 더한 함수를 사용합니다. 상수 이동은 최적점이나 비용 차이를 바꾸지 않으며 비용을 음수가 아니게 만듭니다. 아래 함수의 비볼록 봉우리 때문에 두 국소 후보가 모두 비용을 줄여도 그 평균은 비용을 높일 수 있습니다.

범위는 $0.1\le u_i\le4$, 시작점은 $(0.5,0.5)$입니다. 격자에서 좋은 구간을 찾고 스칼라 최소화로 후보를 보정합니다. 필요한 사실은 후보가 시작점보다 낮은 비용이고 그 평균이 시작점보다 높다는 것입니다. 격자 탐색만으로 일반적인 전역 최적화를 주장하지 않습니다.

$$
J(u)=2+\sum_{i=1}^2(e^{-2u_i}-2e^{-u_i})+1.1e^{-0.4\sum_{i=1}^2(u_i+0.2)^2}
$$

In [ ]:
def ncval(u):
    u = np.asarray(u)
    return float(
        2
        + np.sum(np.exp(-2 * u) - 2 * np.exp(-u))
        + 1.1 * np.exp(-0.4 * np.sum((u + 0.2) ** 2))
    )


def ncgrad(u):
    u = np.asarray(u)
    return (
        -2 * np.exp(-2 * u)
        + 2 * np.exp(-u)
        - 0.8 * (u + 0.2) * 1.1 * np.exp(-0.4 * np.sum((u + 0.2) ** 2))
    )


u0 = np.array([0.5, 0.5])
grid = np.linspace(0.1, 4, 1600)
vgrid = np.array([ncval([v, 0.5]) for v in grid])
j = np.argmin(vgrid)
opt = minimize_scalar(
    lambda v: ncval([v, 0.5]),
    bounds=(grid[max(0, j - 1)], grid[min(len(grid) - 1, j + 1)]),
    method="bounded",
)
assert opt.success
c1 = np.array([opt.x, 0.5])
c2 = c1[::-1]
avg = (c1 + c2) / 2
assert ncval(c1) < ncval(u0) and ncval(c2) < ncval(u0) and ncval(avg) > ncval(u0)
xx, yy = np.meshgrid(np.linspace(0.1, 4, 180), np.linspace(0.1, 4, 180))
Z = (
    2
    + np.exp(-2 * xx)
    - 2 * np.exp(-xx)
    + np.exp(-2 * yy)
    - 2 * np.exp(-yy)
    + 1.1 * np.exp(-0.4 * ((xx + 0.2) ** 2 + (yy + 0.2) ** 2))
)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.7))
axs[0].contour(xx, yy, Z, levels=16, alpha=0.6)
for point, label in [
    (u0, "시작점"),
    (c1, "후보 1"),
    (c2, "후보 2"),
    (avg, "평균 후보"),
]:
    axs[0].scatter(*point, label=label)
axs[0].plot([c1[0], c2[0]], [c1[1], c2[1]], "k--")
axs[0].set(xlabel="입력 1", ylabel="입력 2")
axs[0].legend(fontsize=8)
axs[1].bar(
    ["시작점", "후보 1", "후보 2", "평균 후보"], [ncval(z) for z in [u0, c1, c2, avg]]
)
axs[1].set(ylabel="목적함수 값", ylim=(0.95, 1.25))
CHECKS["비볼록 평균 후보의 비용 증가량"] = ncval(avg) - ncval(u0)
savefig(fig, "nonconvex_average")


### 6.5.2 Distributed Algorithm for Nonconvex Functions — 비볼록 함수의 분산 알고리즘

**교재 인쇄 pp.423–425**

교재의 분산 투영 기울기 방법을 구현합니다. 각 참여자는 자기 블록의 기울기를 구해 허용 집합에 투영하고, Armijo 후퇴 탐색으로 국소 후보를 만듭니다. 후보들을 합친 비용이 가중 후보 비용 이하인지 검사합니다.

검사가 실패하면 비용 개선이 가장 작은 후보를 제외하고 남은 가중치를 다시 정합니다. 끝까지 필요하면 후보 하나를 선택합니다. 이 절의 수렴 목표는 **정상점**이며 전역 최적점 보장이 아닙니다. 세 시작점에서 비용 감소, 입력 범위와 투영 기울기 잔차를 검사합니다.

In [ ]:
def distributed_gradient_step(u, fun, grad, blocks, lo, hi):
    u = np.asarray(u, dtype=float)
    if (
        u.ndim != 1
        or not np.all(np.isfinite(u))
        or not np.isfinite(lo + hi)
        or lo >= hi
    ):
        raise ValueError("유한 입력 벡터와 올바른 상자 범위가 필요합니다.")
    validate_blocks(blocks, u.size)
    if np.any(u < lo) or np.any(u > hi):
        raise ValueError("시작점이 허용 입력 집합 밖에 있습니다.")
    base = fun(u)
    g = np.asarray(grad(u), dtype=float)
    if not np.isfinite(base) or g.shape != u.shape or not np.all(np.isfinite(g)):
        raise ValueError("비용과 기울기는 올바른 차원의 유한 값이어야 합니다.")
    candidates = []
    cv = []
    for idx in blocks:
        direction = np.clip(u[idx] - g[idx], lo, hi) - u[idx]
        slope = g[idx] @ direction
        alpha = 1.0
        for trial in range(60):
            z = u.copy()
            z[idx] += alpha * direction
            value = fun(z)
            if value <= base + 1e-4 * alpha * slope + 1e-14:
                break
            alpha *= 0.5
        else:
            raise RuntimeError("Armijo 탐색 실패")
        candidates.append(z)
        cv.append(value)
    active = list(range(len(blocks)))
    removed = 0
    while True:
        nxt = np.mean([candidates[i] for i in active], axis=0)
        bound = np.mean([cv[i] for i in active])
        if fun(nxt) <= bound + 1e-13:
            break
        active.remove(max(active, key=lambda i: cv[i]))
        removed += 1
        assert active
    assert (
        fun(nxt) <= base + 2e-13
        and np.min(nxt) >= lo - 1e-12
        and np.max(nxt) <= hi + 1e-12
    )
    return nxt, removed


# Analytic gradient versus central finite differences at several interior points.
gap = 0.0
for pt in [np.array([0.5, 0.5]), np.array([1.0, 2.0]), np.array([3.9, 3.6])]:
    fd = np.array(
        [
            (ncval(pt + 1e-6 * np.eye(2)[i]) - ncval(pt - 1e-6 * np.eye(2)[i])) / 2e-6
            for i in range(2)
        ]
    )
    gap = max(gap, np.max(abs(fd - ncgrad(pt))))
assert gap < 1e-8
traces = []
resids = []
drops = 0
for start in [[0.5, 0.5], [3.9, 3.6], [2.99, 3.0]]:
    u = np.array(start)
    tr = [u.copy()]
    for p in range(6000):
        resid = np.linalg.norm(u - np.clip(u - ncgrad(u), 0.1, 4))
        if resid < 1e-7:
            break
        u, drop = distributed_gradient_step(
            u, ncval, ncgrad, [np.array([0]), np.array([1])], 0.1, 4
        )
        drops += drop
        tr.append(u.copy())
    resids.append(np.linalg.norm(u - np.clip(u - ncgrad(u), 0.1, 4)))
    traces.append(np.array(tr))
assert max(resids) < 1e-5
fig, axs = plt.subplots(1, 2, figsize=(11, 3.7))
axs[0].contour(xx, yy, Z, levels=16, alpha=0.4)
for i, tr in enumerate(traces):
    axs[0].plot(tr[:, 0], tr[:, 1], label=f"시작점 {i+1}")
    cost = np.array([ncval(z) for z in tr])
    assert np.diff(cost).max() < 3e-13
    axs[1].plot(cost, label=f"시작점 {i+1}")
axs[0].set(xlabel="입력 1", ylabel="입력 2")
axs[1].set(xlabel="반복 p", ylabel="목적함수 값")
axs[0].legend()
axs[1].legend()
CHECKS["비볼록 비용 기울기 유한차분 최대 차이"] = gap
CHECKS["비볼록 투영 기울기 최대 최종 잔차"] = max(resids)
CHECKS["비볼록 반복의 후보 제외 횟수"] = drops
savefig(fig, "nonconvex_algorithm")


### 6.5.3 Distributed Nonlinear Cooperative Control — 비선형 협조 분산 제어

**교재 인쇄 pp.425–428**

각 참여자가 입력 열의 자기 블록을 개선하도록 앞 절의 방법을 예측 제어 비용에 적용합니다. 보충 모델은 $x^+=0.7x+0.08\tanh(\operatorname{swap}(x))+u$이고 $|u_i|\le0.6$입니다. 비용은 $\sum(\|x\|^2+0.2\|u\|^2)+c\|x_N\|^2$입니다.

영 입력 모델은 전역적으로 $\|f(x,0)\|\le0.78\|x\|$이므로 $c=1/(1-0.78^2)$이면 종단 비용 감소 부등식이 성립합니다. 이 보충 모델에서는 입력 0이 항상 가능하고 상태 강제약을 두지 않으므로 영 입력 종단 정책을 전역적으로 사용할 수 있습니다.

예측 구간은 5이며 각 시간에 반복 1회와 4회를 비교합니다. 기울기는 전체 상태 경로를 통해 역전파해 계산하고 유한차분으로 확인합니다. 두 실행의 성능 차이는 이 예의 결과이며 통신·실시간 병렬 성능 비교가 아닙니다.

In [ ]:
Nn = 5
cn = 1 / (1 - 0.78**2)
Rn = 0.2
blocks = [np.arange(i, 2 * Nn, 2) for i in range(2)]


def nf(x, u):
    return 0.7 * x + 0.08 * np.tanh(x[::-1]) + u


def nj(x):
    return np.array(
        [[0.7, 0.08 / np.cosh(x[1]) ** 2], [0.08 / np.cosh(x[0]) ** 2, 0.7]]
    )


def nvgrad(U, x0):
    us = U.reshape(Nn, 2)
    xs = [x0.copy()]
    for u in us:
        xs.append(nf(xs[-1], u))
    xs = np.array(xs)
    val = np.sum(xs[:-1] ** 2) + Rn * np.sum(us**2) + cn * np.sum(xs[-1] ** 2)
    lam = 2 * cn * xs[-1]
    grad = np.zeros_like(us)
    for k in range(Nn - 1, -1, -1):
        grad[k] = 2 * Rn * us[k] + lam
        lam = 2 * xs[k] + nj(xs[k]).T @ lam
    return float(val), grad.ravel(), xs


rng = np.random.default_rng(6)
utest = rng.uniform(-0.2, 0.2, 2 * Nn)
xtest = np.array([0.9, -0.7])
fd = np.array(
    [
        (
            nvgrad(utest + 1e-6 * np.eye(2 * Nn)[i], xtest)[0]
            - nvgrad(utest - 1e-6 * np.eye(2 * Nn)[i], xtest)[0]
        )
        / 2e-6
        for i in range(2 * Nn)
    ]
)
gap = np.max(abs(fd - nvgrad(utest, xtest)[1]))
assert gap < 1e-7
runs = {}
for iterations in [1, 4]:
    x = np.array([1.5, -1.0])
    warm = np.zeros(2 * Nn)
    history = [x.copy()]
    inputs = []
    values = []
    stages = []
    shifts = []
    for k in range(40):
        fun = lambda U: nvgrad(U, x)[0]
        grad = lambda U: nvgrad(U, x)[1]
        U = warm.copy()
        for p in range(iterations):
            U, _ = distributed_gradient_step(U, fun, grad, blocks, -0.6, 0.6)
        value, _, pred = nvgrad(U, x)
        u = U[:2]
        ell = np.sum(x * x) + Rn * np.sum(u * u)
        xn = nf(x, u)
        wn = np.r_[U[2:], 0.0, 0.0]
        residual = nvgrad(wn, xn)[0] - value + ell
        assert residual < 1e-10
        history.append(xn.copy())
        inputs.append(u.copy())
        values.append(value)
        stages.append(ell)
        shifts.append(residual)
        x = xn
        warm = wn
    history, inputs, values, stages = map(np.array, [history, inputs, values, stages])
    assert np.max(abs(inputs)) <= 0.6 + 1e-12
    assert (
        np.max(np.diff(values) + stages[:-1]) < 1e-10
        and np.linalg.norm(history[-1]) < 1e-4
    )
    runs[iterations] = (history, inputs, values, stages, np.array(shifts))
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for it, (xh, uh, vv, ell, rr) in runs.items():
    axs[0].plot(np.linalg.norm(xh, axis=1), label=f"시간당 {it}회")
    axs[1].step(range(40), uh[:, 0], where="post", label=f"시간당 {it}회")
axs[0].set(xlabel="시간 단계 k", ylabel="실제 상태 크기")
axs[1].set(xlabel="시간 단계 k", ylabel="참여자 1의 실제 입력")
axs[1].axhline(0.6, color="gray", ls="--")
axs[1].axhline(-0.6, color="gray", ls="--")
axs[0].legend()
axs[1].legend()
CHECKS["비선형 MPC 기울기 유한차분 최대 차이"] = gap
CHECKS["비선형 MPC 입력 최대 크기"] = max(np.max(abs(v[1])) for v in runs.values())
savefig(fig, "nonlinear_mpc")


### 6.5.4 Stability of Distributed Nonlinear Cooperative Control — 비선형 협조 제어의 안정성

**교재 인쇄 pp.428–430**

두 종류의 감소를 연결합니다. 먼저 이전 해를 이동하고 종단 정책을 붙인 시작점이 시간 사이의 단계 비용 감소를 제공합니다. 그 시작점에서의 분산 최적화는 비용을 더 낮춥니다. 실행 가능성, 적절한 종단 조건과 양의 정부호 비용 등 정리의 가정이 함께 필요합니다.

보충 모델에서는 $c(0.78^2-1)+1=0$이므로 영 입력 종단 부등식의 전역 경계를 해석적으로 확인할 수 있습니다. 앞 절에서 생성한 두 제어 경로의 시간 간 비용 감소 잔차를 검사합니다. 수치 경로 검사와 일반적인 안정성 정리의 증명은 구분합니다.

In [ ]:
terminal_bound = cn * (0.78**2 - 1) + 1
assert abs(terminal_bound) < 1e-12
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
for it, (xh, uh, vv, ell, rr) in runs.items():
    residual = np.diff(vv) + ell[:-1]
    assert residual.max() < 1e-10
    axs[0].semilogy(np.maximum(vv, 1e-16), label=f"시간당 {it}회")
    axs[1].plot(residual, label=f"시간당 {it}회")
    CHECKS[f"비선형 MPC {it}회 반복 최종 상태 크기"] = np.linalg.norm(xh[-1])
    CHECKS[f"비선형 MPC {it}회 반복 비용 감소 최대 잔차"] = residual.max()
    CHECKS[f"비선형 MPC {it}회 반복 누적 단계 비용"] = ell.sum()
axs[0].set(xlabel="시간 단계 k", ylabel="실현된 예측 비용")
axs[1].set(xlabel="시간 단계 k", ylabel="다음 비용 − 현재 비용 + 단계 비용")
axs[0].legend()
axs[1].legend()
axs[1].axhline(0, color="gray", ls="--")
CHECKS["비선형 종단 부등식 계수 잔차"] = abs(terminal_bound)
savefig(fig, "nonlinear_stability")


#### 원본 예제 6.22: 불안정 비선형 모델의 첫 최적화

**원본 인쇄 pp.429–432, Figure 6.10의 한 단계 비용**을 확인합니다. 앞서 사용한 수축하는 보충 모델과 달리 원본은
$$x_1^+=x_1^2+x_2+u_1^3+u_2,\qquad x_2^+=x_1+x_2^2+u_1+u_2^3$$
이며 초기 상태는 $(3,-3)$입니다. 원본과 같은 $Q=R=I$, 입력 상자 $[-2.5,2.5]^2$를 사용합니다. 원점의 선형화 $A=B=[0,1;1,0]$에서 DARE 해 $P$를 구해 $V_f=\frac12x^TPx$를 둡니다.

**범위는 초기 상태에서 N=1인 비용 지형과 분산 개선 경로**입니다. 시작 입력은 여기서 $(0,0)$으로 정했습니다. Figure 6.9의 N=2 폐루프나 원본의 종단 영역을 재현한 실험은 아닙니다. 특히 원본의 불안정 모델에 앞선 보충 예의 영 입력 종단 조건을 적용하면 안 됩니다. 원점 선형화의 LQR만으로 비선형 전역 안정성도 결론지을 수 없습니다.

코드는 해석 기울기를 중앙 유한차분과 비교하고 후보 검사 후의 비용 감소를 검증합니다. 아래 경로는 전역 최적점 보장이 아닌 유한 반복의 개선 결과입니다.

In [ ]:
linear = np.array([[0.0, 1.0], [1.0, 0.0]])
p22 = solve_discrete_are(linear, linear, np.eye(2), np.eye(2))
x22 = np.array([3.0, -3.0])


def plant22(x, u):
    return x * x + x[::-1] + u**3 + u[::-1]


def cost22(u):
    nxt = plant22(x22, u)
    return float(0.5 * (x22 @ x22 + u @ u + nxt @ p22 @ nxt))


def grad22(u):
    nxt = plant22(x22, u)
    jac = np.diag(3 * u * u) + linear
    return u + jac.T @ p22 @ nxt


point22 = np.array([-0.8, -1.3])
basis = np.eye(2)
eps22 = 1e-6
fd22 = np.array(
    [
        (cost22(point22 + eps22 * v) - cost22(point22 - eps22 * v)) / (2 * eps22)
        for v in basis
    ]
)
gap22 = np.max(abs(fd22 - grad22(point22)))
assert gap22 < 1e-6
u22 = np.zeros(2)
path22 = [u22.copy()]
costs22 = [cost22(u22)]
for _ in range(30):
    u22, _ = distributed_gradient_step(
        u22, cost22, grad22, [np.array([0]), np.array([1])], -2.5, 2.5
    )
    path22.append(u22.copy())
    costs22.append(cost22(u22))
path22 = np.array(path22)
assert np.max(np.diff(costs22)) < 1e-10 and costs22[-1] < costs22[0]
g1, g2 = np.meshgrid(np.linspace(-2.5, 2.5, 180), np.linspace(-2.5, 2.5, 180))
n1 = 6 + g1**3 + g2
n2 = 12 + g1 + g2**3
landscape = 0.5 * (
    18
    + g1 * g1
    + g2 * g2
    + p22[0, 0] * n1 * n1
    + 2 * p22[0, 1] * n1 * n2
    + p22[1, 1] * n2 * n2
)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.7))
axs[0].contour(
    g1, g2, landscape, levels=[15, 20, 30, 50, 100, 200, 400, 800], alpha=0.6
)
axs[0].plot(path22[:, 0], path22[:, 1], "o-", markersize=3, label="30회 개선")
axs[0].scatter([0], [0], marker="x", s=65, label="시작 입력")
axs[0].set(xlabel="입력 1", ylabel="입력 2")
axs[0].legend(fontsize=9)
axs[1].semilogy(costs22)
axs[1].set(xlabel="최적화 반복 p", ylabel="N=1 예측 비용")
CHECKS["원본 6.22 초기 비용 기울기 유한차분 차이"] = gap22
CHECKS["원본 6.22 첫 최적화 초기 비용"] = costs22[0]
CHECKS["원본 6.22 첫 최적화 30회 후 비용"] = costs22[-1]
savefig(fig, "textbook_nonlinear")


## 6.6 Notes — 문헌 해설

**교재 인쇄 pp.430–434**

교재는 게임 이론, 분산 최적화, 제어 이론의 기여를 연결합니다. 내시점, 전체 최적해, 정상점, 제어 안정성은 다른 개념이므로 결과를 읽을 때 무엇이 보장되는지 확인해야 합니다. 공유 자원이나 비선형성에 대한 연구 현황 설명은 원본 2017년 판의 문맥으로 읽습니다.

## 6.7 Exercises — 연습문제

**교재 인쇄 pp.435–447**

교재 연습문제 6.1은 LQ 문제의 세 계산법, 6.22는 입력 열 크기 제한의 선형 부등식 표현, 6.29는 외란 모델의 검출가능성을 다룹니다. 이번 노트북의 보충 과제로 계산 구조와 가정을 바꾸어 보세요. 모든 교재 연습문제의 해답을 포함하지 않습니다.

## 보충 과제와 힌트

1. 분권 제어의 국소 모델을 고정하고 입력 결합 크기를 바꾸면 전체 고유값은 어떻게 달라지나요?
2. 비협조 내시점이 존재하는데도 동시 최선 반응이 발산하는 이유를 반복 행렬로 설명하세요.
3. 협조 갱신에서 전체 후보 평균을 각자의 새 입력 단순 결합으로 바꾸면 어떤 성질을 잃을 수 있나요?
4. 공유 자원 예의 경계점에서 두 참여자가 함께 개선할 수 있는 방향을 구하세요.
5. 비볼록 후보 검사에서 탈락 규칙을 제거하고 항상 평균하면 비용 감소가 유지되나요?
6. 비선형 MPC의 끝 입력을 0 대신 임의의 값으로 바꾸면 이동 시작점 비용 부등식이 유지되나요?

<details><summary>힌트</summary>

1. $A+B K_d$의 고유값을 직접 계산하세요. 특정 초기 상태 경로만 보지 마세요.
2. 내시점은 연립식의 해 존재 문제이고, 최선 반응 반복은 $\rho(T_{BR})<1$ 여부의 문제입니다.
3. 볼록 결합의 가중치 합이 1이어야 합니다. 새 블록들을 모두 붙이면 공유 자원 제약을 위반할 수도 있습니다.
4. $u_1+u_2=1$을 유지하면서 참여자 1의 자원을 참여자 2에게 옮기는 방향을 보세요.
5. §6.5.1의 비용 증가 반례를 다시 확인하세요. 실제 후보의 비용 평가가 필요합니다.
6. 종단 비용 감소 조건에 사용한 정책이 0이므로 그 부등식을 다시 검증해야 합니다.

</details>

## 실행 검증 결과

수치 해 일치, 반복 수렴, 실제 폐루프 고유값, 제약 만족과 시간 간 비용 감소를 구분해 기록합니다.

In [ ]:
for name, value in CHECKS.items():
    print(f"{name}: {value:.8g}")
print("검증 지표 수:", len(CHECKS))
print(
    {
        "Python": sys.version.split()[0],
        "NumPy": np.__version__,
        "SciPy": scipy.__version__,
        "Matplotlib": matplotlib.__version__,
        "CVXPY": cp.__version__,
    }
)


## 참고 자료

- Rawlings, Mayne, Diehl (2017), *Model Predictive Control: Theory, Computation, and Design*, 2nd edition, 1st printing, Chapter 6, pp.369–450.
- 선형 대수와 유한 반복의 기억은 §6.1, 네 제어 방식과 목표·추정은 §6.2, 독립·결합 제약은 §6.3을 참고합니다.
- §6.5.1–6.5.2는 교재 그림 6.7–6.8의 목적함수와 투영 기울기·Armijo·후보 제외 방식을 사용합니다. 상수 2를 더했으며 실행 범위·시작점·종료 조건을 명시했습니다.
- 예제 6.9–6.11은 원본 전달함수와 설계값에서 반복·폐루프 고유값을 재현합니다. 예제 6.12는 두 짝짓기의 목표 반복을 재현합니다. 예제 6.22는 원본 동역학의 초기 N=1 비용 지형을 분석하며 N=2 폐루프 재현은 아닙니다.
- 다른 실행 예제는 직접 설계한 보충 실습입니다. 실제 통신이나 다중 장치의 병렬 실행을 수행하는 프로그램은 아닙니다.
- 한글 글꼴은 Google Fonts의 나눔고딕을 사용합니다.